# DataFrameReader and DataFrameWriter Comprehensive Reference

**Module**: 08_data_sources_and_formats  
**Notebook**: 05_reading_and_writing_data.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The complete reference guide to DataFrameReader and DataFrameWriter APIs: formats, modes (append, overwrite, error, ignore), bucketing, sorting, and small file control.
- **Why does this topic exist?**: Data ingestion and egress are the boundaries of every Spark pipeline.
- **What problem does it solve?**: Accidentally wiping tables with wrong save modes or creating millions of output files.
- **Where is it used in real-world Data Engineering?**: Building resilient production ETL jobs.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Data sources, schemas, partitioning |
| **Required depth** | Working knowledge: can use reader/writer options and choose safe save semantics |
| **Data Engineer** | Write data with the right mode, partitioning, and table semantics. |
| **Senior Data Engineer** | Prevent accidental overwrites and design idempotent writes. |
| **Spark specialist** | Reason about table catalog behavior, insertInto versus saveAsTable, and transactional sinks. |
| **Next topic** | `09_streaming/01_streaming_fundamentals.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

The reader and writer APIs look simple, but the defaults matter: `overwrite` can delete data, `append` can
duplicate it, and `saveAsTable` versus `insertInto` follow different schema and catalog rules. A production write
must be idempotent, partition-aware, and explicit about what happens when the target already exists.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Save mode** | Behavior when target exists | append, overwrite, errorifexists, ignore |
| **saveAsTable** | Create/use a catalog table | Uses table schema and catalog semantics |
| **insertInto** | Insert into existing table | Matches by position and ignores column names in some cases |
| **partitionBy** | Physical layout | Creates partition directories on write |
| **bucketBy** | Hash layout | Creates fixed buckets for repeated joins |
| **overwritePartitions** | Dynamic overwrite | Replaces only partitions present in the new data |

### How it works, step by step

1. The writer evaluates the DataFrame and plans the output.
2. It checks the target and applies the save mode.
3. Partition columns are extracted and used to create directories.
4. Files are written per partition, with one file per task by default.
5. The catalog is updated for table writes.
6. A failure mid-write can leave partial output unless the sink is transactional.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Save Modes: Append, Overwrite, ErrorIfExists, Ignore

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Read_Write_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")

# DataFrameWriter options: mode, format, partitionBy, coalesce
out = "/opt/spark-data/warehouse/orders_writer_test"
(
    orders
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(out)
)
print("Successfully written with DataFrameWriter!")

---

## Experiment: SaveMode Overwrite vs Append

- **1. What are we testing?**: Verify row count when writing in append mode vs overwrite mode.
- **2. What do we expect?**: Append doubles the row count; Overwrite replaces it.

In [ ]:
orders.limit(5).write.mode("append").parquet(out)
print(f"Total count after append: {spark.read.parquet(out).count()}")

**3. What happened?**: New rows appended alongside existing files.  
**4. Why did it happen?**: Append mode creates new part files without removing old ones.  
**5. Practical Engineering Lesson**: Use overwrite mode for idempotent batch reprocessing; use append for incremental loads.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Idempotent daily load

- **Industry**: Retail
- **Scenario**: A daily job appends sales data and is sometimes rerun after a failure.
- **Problem**: A plain append duplicates the day's rows on every retry.
- **Why this concept is relevant**: Writing to a staging location and replacing the target partition, or using a transactional table format, makes retries idempotent.
- **Implementation idea**: Use dynamic partition overwrite or a merge/upsert pattern keyed by date and order ID.
- **Expected result**: Exactly one copy of each day's data after any number of retries.
- **Alternative**: Deduplicate downstream with a window function.
- **Trade-offs**: Downstream dedup hides the problem but wastes storage and compute and can still be wrong.

### Use Case 2: Exporting for a warehouse

- **Industry**: Logistics
- **Scenario**: A Spark job exports a curated table to a JDBC warehouse.
- **Problem**: The export fails halfway and the warehouse contains a partial load.
- **Why this concept is relevant**: Batching with a staging table and a transactional swap prevents partial visibility.
- **Implementation idea**: Write to a staging table, validate counts, then swap or merge in the warehouse.
- **Expected result**: A warehouse that never exposes a partial batch.
- **Alternative**: Use the warehouse's native loader.
- **Trade-offs**: A native loader may be faster but adds coupling to the warehouse vendor.

---

## Business Value

- No duplicate or partial data after retries.
- Predictable catalog and schema behavior.
- Lower storage waste from repeated appends.
- Auditable load semantics.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Prefer dynamic partition overwrite for partition-scoped backfills.
- Use `saveAsTable` when you need catalog semantics and a stable schema.
- Be careful with `insertInto`: column order matters.
- Make writes idempotent before adding retries.
- Validate row counts and keys after every load.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Make a write idempotent

**Steps**

1. Append the same batch twice and count duplicate keys.
2. Switch to a dynamic partition overwrite or a merge keyed by the batch ID.
3. Rerun the batch and verify the count.

**What to observe**: A naive append duplicates rows; a partition-scoped overwrite or merge keeps one copy.

**Healthy result**: The key count is stable across reruns.

**Problematic result**: The target has partial files after a failure.

**Engineering takeaway**: Idempotency is a write-design requirement, not a retry setting.

---

## Trade-offs and Alternatives

### Advantages

- Rich options
- Works across formats and databases
- Supports partitioning and bucketing

### Limitations

- Defaults can be destructive
- Non-transactional file writes
- Catalog semantics vary

### When to use it

- Batch exports
- Lake ingestion
- Table maintenance

### When not to use it

- Blind overwrite
- Append-only retries
- Mixing insertInto and saveAsTable assumptions

### Alternatives inside Spark

- DataFrameWriter
- Save modes
- Dynamic overwrite

### Alternatives outside Spark

- Transactional table formats
- Warehouse loaders
- CDC pipelines

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Read with Format
**Problem Statement**: Read using spark.read.format('parquet').load(path).  
**Input / Context**: out

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Save As Table
**Problem Statement**: Explain saveAsTable vs save.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Dynamic Partition Overwrite
**Problem Statement**: Set dynamic partition overwrite mode.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The overwrite that removed history

**Scenario**: A monthly job writes with `mode('overwrite')` and the table now contains only the latest month.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the write mode and target path.
2. Determine whether dynamic partition overwrite was intended.
3. Restore from backup and rewrite with partition-scoped semantics.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Read with Format):
```python
spark.read.format('parquet').load(out).show(2)
```

#### Solution 2 (Save As Table):
```python
print('saveAsTable registers table metadata in the Metastore catalog.')
```

#### Solution 3 (Dynamic Partition Overwrite):
```python
spark.conf.set('spark.sql.sources.partitionOverwriteMode','dynamic')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Reader/Writer**: Unified API across CSV, JSON, Parquet, ORC, JDBC.
- **Idempotency**: Use overwrite mode for reproducible batch pipelines.

### Essential Technical Terms
- `SaveMode`: Append, Overwrite, Ignore, ErrorIfExists.
- `Dynamic Partition Overwrite`: Overwrites only touched partition folders.

### Common Engineering Mistakes
- **Uncontrolled partition output**: Writing without coalesce/repartition produces 200 tiny files.

### Interview Questions on This Topic
- **Q: What are the 4 save modes in DataFrameWriter?**
  - *A*: Append, Overwrite, ErrorIfExists (default), and Ignore.


### More Interview Questions

- **Q: What is the difference between `saveAsTable` and `insertInto`?**
  - *A*: `saveAsTable` creates or replaces a catalog table using the DataFrame schema; `insertInto` writes into an existing table and generally matches columns by position, so schema and order must align.
- **Q: Why is a plain append not idempotent?**
  - *A*: Rerunning the job appends the same records again, producing duplicates unless the data is deduplicated or the write is scoped to a partition/merge key.

### What to Study Next
Module 08 complete! Proceed to Module 09: Structured Streaming.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')